# 05 — EXP_07: Adaptive RAG (rule-based router → 5 underlying architectures)

**Architecture:** per-question routing using the EXP_06 complexity labels (`Simple` / `Moderate` / `Complex`). Two routing tables run side-by-side:

| Bucket | **Variant A** (proposal) | **Variant B** (data-driven) |
|---|---|---|
| Simple | NaiveRetriever (k=5 dense) | NoRetrieval (No-RAG) |
| Moderate | HybridRetriever (RRF k=60) | MultiHopRetriever (3 hops) |
| Complex | MultiHopRetriever | MultiHopRetriever |

**Generator:** `llama-3.3-70b-versatile` via Groq · T=0 · max_tokens=700.

## Key efficiency property

Because EXP_02–05 all ran with deterministic retrievers + the same prompts at T=0, **most of the EXP_07 Groq calls hit the disk cache**. Per-variant wall time on test_1273 should be ~5–15 min (vs ~30–45 min if cache cold).

## Falsifiable hypotheses for EXP_07

Anchored in the Phase 4 close-out + EXP_06 simulator. The naive `Acc / calls/Q` ratio is misleading because it trivially favours the cheapest architecture (No-RAG sits at 0.7738 / 1.00 = 0.7738). The honest framing is the **Pareto frontier**: a routing strategy earns its keep if it sits on the accuracy-vs-compute frontier (no other strategy strictly dominates it).

1. **Variant A is on the Pareto frontier** between No-RAG and all-Multi-Hop — strictly higher Acuuracy than No-RAG, strictly lower compute than all-Multi-Hop. Falsified if Variant A is dominated (some other point with both higher acc AND lower compute exists).
2. **Variant A dominates Variant B** — i.e. Variant A has both higher Acuuracy AND fewer Groq calls per question than Variant B. Falsified if Variant B is on the frontier (better acc OR fewer calls than A).
3. **Variant A's marginal accuracy gain over No-RAG is meaningfully larger than Multi-Hop's marginal gain over Variant A** (i.e., adaptive routing captures most of the bang-for-buck). Falsified if Multi-Hop's marginal Acc/extra-call > Variant A's marginal Acc/extra-call vs No-RAG.

## Stages

| Stage | Variant | Surface | Wall time | Cost |
|---|---|---|---|---|
| **A** Smoke | A | 50 stratified | ~3 min | $0 |
| **B** Smoke | B | 50 stratified | ~3 min | $0 |
| **C** Full | A | test_1273 | ~5–15 min (cached) | $0 |
| **D** Full | B | test_1273 | ~5–15 min (cached) | $0 |
| **E** RAGAS join | both | golden_234 | <1 min | $0 |

## 1. Setup

In [1]:
import json
import logging
import os
import sys
from pathlib import Path

os.environ["ANONYMIZED_TELEMETRY"] = "False"
logging.getLogger("chromadb").setLevel(logging.WARNING)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

import pandas as pd

from src.data.embedder import best_device, load_bge
from src.data.indices import load_bm25, load_chroma
from src.data.loaders import load_chunks, load_medqa_4opt
from src.eval.runner import run_experiment
from src.retrieval.adaptive import AdaptiveRetriever
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.multi_hop import MultiHopRetriever
from src.retrieval.naive import NaiveRetriever
from src.retrieval.none import NoRetrieval

print("GROQ_API_KEY present:", "\u2713" if os.environ.get("GROQ_API_KEY") else "\u2717")
print("Repo root:", REPO_ROOT)
print("Device   :", best_device())

GROQ_API_KEY present: ✓
Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Device   : mps


## 2. Build underlying retrievers + question→bucket lookup

In [2]:
CHROMA_DIR = REPO_ROOT / "data" / "indices" / "chroma_textbooks"
BM25_PATH = REPO_ROOT / "data" / "indices" / "bm25.pkl"

chroma_coll = load_chroma(CHROMA_DIR)
bm25_payload = load_bm25(BM25_PATH)
embedder = load_bge(device=best_device())
chunks_df = load_chunks()

# 5 underlying retrievers — same configurations as EXP_01–05
no_rag = NoRetrieval()
naive = NaiveRetriever(chroma_coll, embedder)
hybrid = HybridRetriever(embedder, chroma_coll, bm25_payload, chunks_df)
multi_hop = MultiHopRetriever(embedder, chroma_coll, max_hops=3, per_hop_k=5)

# Build the question-text → bucket lookup from EXP_06 + medqa_4opt
labels = pd.read_parquet(REPO_ROOT / "data/processed/complexity_labels.parquet")
md = pd.read_parquet(REPO_ROOT / "data/processed/medqa_4opt.parquet")
md = md.reset_index(drop=False).rename(columns={"index": "row_idx"})
md["question_id"] = "medqa_" + md["row_idx"].astype(str)
joined = labels.merge(md[["question_id", "question"]], on="question_id")
Q_TO_BUCKET = dict(zip(joined["question"], joined["complexity"].astype(str)))

print(f"ChromaDB count : {chroma_coll.count():,}")
print(f"BM25 chunks    : {len(bm25_payload['chunk_ids']):,}")
print(f"chunks.parquet : {len(chunks_df):,}")
print(f"Lookup size    : {len(Q_TO_BUCKET):,} questions")
print(f"Bucket counts  : {pd.Series(list(Q_TO_BUCKET.values())).value_counts().to_dict()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ChromaDB count : 67,599
BM25 chunks    : 67,599
chunks.parquet : 67,599
Lookup size    : 12,721 questions
Bucket counts  : {'Complex': 4799, 'Moderate': 4163, 'Simple': 3759}


## 3. Define the two routing tables

In [3]:
VARIANT_A = AdaptiveRetriever(
    Q_TO_BUCKET,
    {"Simple": naive, "Moderate": hybrid, "Complex": multi_hop},
)
VARIANT_B = AdaptiveRetriever(
    Q_TO_BUCKET,
    {"Simple": no_rag, "Moderate": multi_hop, "Complex": multi_hop},
)

# Sanity check — pick one Simple, one Moderate, one Complex question and
# verify each variant routes correctly.
for label in ("Simple", "Moderate", "Complex"):
    q = next(q for q, b in Q_TO_BUCKET.items() if b == label)
    print(f"\n[{label}] {q[:90]}...")
    print(f"  Variant A bucket: {VARIANT_A.bucket_for(q)}")
    print(f"  Variant B bucket: {VARIANT_B.bucket_for(q)}")


[Simple] A 3-month-old baby died suddenly at night while asleep. His mother noticed that he had die...
  Variant A bucket: Simple
  Variant B bucket: Simple

[Moderate] A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. S...
  Variant A bucket: Moderate
  Variant B bucket: Moderate

[Complex] A 40-year-old zookeeper presents to the emergency department complaining of severe abdomin...
  Variant A bucket: Complex
  Variant B bucket: Complex


## 4. Configuration

In [6]:
RESULTS_DIR = REPO_ROOT / "results"
TOP_K = 15  # multi_hop's max chunk return; naive/hybrid/no_rag still cap at their own k internally

SMOKE_N = 50
SMOKE_SEED = 42

RUN_SMOKE = True
RUN_FULL_A = True
RUN_FULL_B = True

## 5. Stage A — Smoke (50 stratified rows · both variants)

The same 50 questions go through both variants — same seed as EXP_02–05 smoke for comparability.

In [7]:
if RUN_SMOKE:
    medqa = load_medqa_4opt()
    smoke = (
        medqa.groupby("meta_info", group_keys=False)
        .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))
        .reset_index(drop=True)
    )
    smoke = smoke.head(SMOKE_N) if len(smoke) >= SMOKE_N else medqa.sample(n=SMOKE_N, random_state=SMOKE_SEED).reset_index(drop=True)
    print(f"Smoke surface: {len(smoke)} rows | meta_info mix: {dict(smoke['meta_info'].value_counts())}")

    for variant_label, variant_retriever in (("A", VARIANT_A), ("B", VARIANT_B)):
        out_dir = RESULTS_DIR / f"exp_07_adaptive_variant_{variant_label.lower()}__smoke_50"
        summary = run_experiment(
            retriever=variant_retriever,
            dataset=smoke,
            output_dir=out_dir,
            experiment_id=f"EXP_07_ADAPTIVE_VARIANT_{variant_label}",
            dataset_label="smoke_50",
            k=TOP_K,
        )
        print(f"\n--- Variant {variant_label} smoke ---")
        print(f"  Acuuracy   : {summary['Acuuracy']:.4f}")
        print(f"  wall_time  : {summary['wall_time_s_this_run']:.1f} s")
        print(f"  cache_hits : {summary['cache_hits_this_run']}/{summary['n_calls_this_run']} ({summary['cache_hit_rate_this_run']*100:.1f} %)")
        print(f"  dispatch   : {variant_retriever.dispatch_counts}")
        print(f"  unknown qs : {variant_retriever.unknown_question_count}")
else:
    print("RUN_SMOKE = False — skipping Stage A")

Smoke surface: 50 rows | meta_info mix: {'step1': np.int64(28), 'step2&3': np.int64(22)}


/var/folders/q2/0276zgxs0m7bj83vqzpkxlqh0000gn/T/ipykernel_19842/1998463249.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=max(1, round(SMOKE_N * len(g) / len(medqa))), random_state=SMOKE_SEED))


EXP_07_ADAPTIVE_VARIANT_A:   0%|          | 0/50 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



--- Variant A smoke ---
  Acuuracy   : 0.8200
  wall_time  : 79.7 s
  cache_hits : 16/50 (32.0 %)
  dispatch   : {'Simple': 17, 'Moderate': 17, 'Complex': 16}
  unknown qs : 0


EXP_07_ADAPTIVE_VARIANT_B:   0%|          | 0/50 [00:00<?, ?it/s]


--- Variant B smoke ---
  Acuuracy   : 0.8600
  wall_time  : 4.8 s
  cache_hits : 50/50 (100.0 %)
  dispatch   : {'Simple': 17, 'Moderate': 17, 'Complex': 16}
  unknown qs : 0


**Acceptance gates (Stage A)** — both variants should: (1) produce 50 predictions; (2) have **high cache hit rate** (>70 %) since the underlying retrievers are deterministic and EXP_02–05 already populated the cache; (3) have `unknown_question_count == 0` (lookup miss = bug); (4) Variant A Acuuracy ≥ 0.74, Variant B ≥ 0.72.

## 6. Stage C — Full test_1273 · Variant A (proposal)

In [8]:
if RUN_FULL_A:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    summary_a = run_experiment(
        retriever=VARIANT_A,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_07_adaptive_variant_a__test_1273",
        experiment_id="EXP_07_ADAPTIVE_VARIANT_A",
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(summary_a, indent=2))
    print(f"\nVariant A dispatch: {VARIANT_A.dispatch_counts}")
    print(f"Variant A unknown : {VARIANT_A.unknown_question_count}")
else:
    print("RUN_FULL_A = False — skipping Stage C")

Test-split surface: 1273 rows


EXP_07_ADAPTIVE_VARIANT_A:   0%|          | 0/1273 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{
  "experiment_id": "EXP_07_ADAPTIVE_VARIANT_A",
  "dataset": "test_1273",
  "n_questions": 1273,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.786331500392773,
  "Exact_Match": 0.786331500392773,
  "n_correct": 1001,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.6963811892743946,
  "wall_time_s_this_run": 1829.174992799759,
  "n_calls_this_run": 1273,
  "cache_hits_this_run": 509,
  "cache_hit_rate_this_run": 0.39984289080911234,
  "timestamp_utc": "2026-05-11T11:18:58Z"
}

Variant A dispatch: {'Simple': 383, 'Moderate': 411, 'Complex': 529}
Variant A unknown : 0


## 7. Stage D — Full test_1273 · Variant B (data-driven binary)

In [9]:
if RUN_FULL_B:
    medqa = load_medqa_4opt()
    medqa = medqa[medqa["split"] == "test"].reset_index(drop=True)
    print(f"Test-split surface: {len(medqa)} rows")

    summary_b = run_experiment(
        retriever=VARIANT_B,
        dataset=medqa,
        output_dir=RESULTS_DIR / "exp_07_adaptive_variant_b__test_1273",
        experiment_id="EXP_07_ADAPTIVE_VARIANT_B",
        dataset_label="test_1273",
        k=TOP_K,
    )
    print(json.dumps(summary_b, indent=2))
    print(f"\nVariant B dispatch: {VARIANT_B.dispatch_counts}")
    print(f"Variant B unknown : {VARIANT_B.unknown_question_count}")
else:
    print("RUN_FULL_B = False — skipping Stage D")

Test-split surface: 1273 rows


EXP_07_ADAPTIVE_VARIANT_B:   0%|          | 0/1273 [00:00<?, ?it/s]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{
  "experiment_id": "EXP_07_ADAPTIVE_VARIANT_B",
  "dataset": "test_1273",
  "n_questions": 1273,
  "Generator_Model": "llama-3.3-70b-versatile",
  "temperature": 0.0,
  "max_tokens": 700,
  "Acuuracy": 0.7831893165750197,
  "Exact_Match": 0.7831893165750197,
  "n_correct": 997,
  "Recall@3": null,
  "Recall@5": null,
  "Recall@10": null,
  "MRR": null,
  "RAGAS_Faithfulness": null,
  "RAGAS_Hallucination_Rate": null,
  "RAGAS_Answer_Relevance": null,
  "RAGAS_Context_Precision": null,
  "RAGAS_Context_Recall": null,
  "Answer_Correctness": null,
  "mean_latency_s": 0.5739680656933541,
  "wall_time_s_this_run": 136.54666209220886,
  "n_calls_this_run": 1273,
  "cache_hits_this_run": 1264,
  "cache_hit_rate_this_run": 0.992930086410055,
  "timestamp_utc": "2026-05-11T11:24:39Z"
}

Variant B dispatch: {'Simple': 383, 'Moderate': 411, 'Complex': 529}
Variant B unknown : 0


## 8. Stage E — RAGAS score-join from underlying golden_234 results

**Why score-join, not re-run RAGAS**: Adaptive routes per question, so its golden_234 RAGAS metrics are by construction a per-question average of whichever underlying architecture's score applies to that question. Re-running the judge would cost ~$11–13 to compute exactly the same numbers.

Algorithm: for each golden_234 question, look up its bucket; pull that bucket's underlying-architecture per-row RAGAS score; average across the 234 rows.

In [10]:
RAGAS_METRICS = [
    "faithfulness",
    "context_precision",
    "context_recall",
    "answer_relevancy",
    "answer_correctness",
]

# Map underlying arch → its golden_234 ragas_scores.csv path
RAG_PATHS = {
    "NoRAG": REPO_ROOT / "results/exp_01_base_llm__golden_234/ragas_scores.csv",
    "Naive": REPO_ROOT / "results/exp_02_naive_rag__golden_234/ragas_scores.csv",
    "Hybrid": REPO_ROOT / "results/exp_04_hybrid_rag__golden_234/ragas_scores.csv",
    "MultiHop": REPO_ROOT / "results/exp_05_multi_hop_rag__golden_234/ragas_scores.csv",
}

ragas_per_arch = {}
for arch, p in RAG_PATHS.items():
    if not p.exists():
        print(f"  skipping {arch} — {p} not found")
        continue
    df = pd.read_csv(p)
    # EXP_01 only has answer_correctness + answer_relevancy — others are NaN by design
    ragas_per_arch[arch] = df.set_index("question_id")

# Build the bucket label per golden_234 question (golden ids are 'golden_NNN'; we
# need to look them up in our medqa_4opt-anchored Q_TO_BUCKET via question text).
import json as _json
golden = pd.DataFrame([_json.loads(l) for l in (REPO_ROOT/"data/processed/golden_ragas_300.jsonl").read_text().splitlines()])
golden["question_id"] = golden.question_id.apply(lambda i: f"golden_{int(i):03d}")
golden["bucket"] = golden.question.map(Q_TO_BUCKET)
missing_buckets = golden.bucket.isna().sum()
print(f"Golden 234 bucket join: {len(golden)} rows, {missing_buckets} missing")
if missing_buckets:
    print("  missing-bucket question_ids:", golden[golden.bucket.isna()].question_id.tolist())
print(f"  bucket counts: {golden.bucket.value_counts().to_dict()}")

ROUTING_TABLES = {
    "A": {"Simple": "Naive", "Moderate": "Hybrid", "Complex": "MultiHop"},
    "B": {"Simple": "NoRAG", "Moderate": "MultiHop", "Complex": "MultiHop"},
}

def score_join(routing: dict[str, str]) -> dict[str, float]:
    """Per-question pick from underlying arch RAGAS scores, then mean."""
    out = {m: [] for m in RAGAS_METRICS}
    for _, g in golden.iterrows():
        bucket = g["bucket"]
        if bucket not in routing:
            continue
        arch = routing[bucket]
        if arch not in ragas_per_arch:
            continue
        rag_row = ragas_per_arch[arch].loc[g.question_id] if g.question_id in ragas_per_arch[arch].index else None
        if rag_row is None:
            continue
        for m in RAGAS_METRICS:
            v = rag_row.get(m)
            if v is not None and pd.notna(v):
                out[m].append(float(v))
    return {m: (sum(vs)/len(vs) if vs else float("nan")) for m, vs in out.items()}

for label, table in ROUTING_TABLES.items():
    scores = score_join(table)
    print(f"\n=== Variant {label} score-joined RAGAS (golden_234, route per Q) ===")
    for m, v in scores.items():
        print(f"  {m:25s} : {v:.4f}")

Golden 234 bucket join: 234 rows, 0 missing
  bucket counts: {'Complex': 108, 'Simple': 63, 'Moderate': 63}

=== Variant A score-joined RAGAS (golden_234, route per Q) ===
  faithfulness              : 0.1966
  context_precision         : 0.3598
  context_recall            : 0.5705
  answer_relevancy          : 0.5971
  answer_correctness        : 0.8473

=== Variant B score-joined RAGAS (golden_234, route per Q) ===
  faithfulness              : 0.2756
  context_precision         : 0.3792
  context_recall            : 0.7544
  answer_relevancy          : 0.5932
  answer_correctness        : 0.8669


## 9. Inspect — Pareto frontier on accuracy vs Groq calls

Naive `Acc / calls/Q` favours No-RAG trivially (it has the fewest calls). The honest comparison is the **Pareto frontier** — which strategies are not dominated by any other?

Per-architecture Groq calls per question:
- NoRAG, Naive, Sparse, Hybrid: **1** (one answer-generation call)
- MultiHop: **3** (1 final answer + ≤2 sub-queries on hops 2/3)
- Adaptive variants: weighted average over the bucket → architecture mapping

A strategy is on the **Pareto frontier** if no other strategy has both higher accuracy AND fewer Groq calls. The frontier defines the menu of cost-quality trade-offs — choosing among them is a deployment decision, not an empirical one.

The discussion-chapter claim: **Variant A is the middle-of-the-frontier choice — it captures most of the marginal accuracy gain over No-RAG, and the additional gain from all-Multi-Hop comes at materially worse marginal efficiency.**

In [11]:
# Per-arch calls/Q (constants from architecture)
CALLS_PER_Q = {
    "NoRAG": 1,
    "Naive": 1,
    "Sparse": 1,
    "Hybrid": 1,
    "MultiHop": 3,
}

def variant_calls_per_q(routing: dict[str, str], bucket_counts: dict[str, int]) -> float:
    total = sum(bucket_counts.values())
    return sum(bucket_counts[b] * CALLS_PER_Q[arch] for b, arch in routing.items()) / total

# Gather published / produced numbers
headline = {}
for prefix, label, calls in [
    ("exp_01_base_llm", "NoRAG", 1.0),
    ("exp_02_naive_rag", "Naive", 1.0),
    ("exp_03_sparse_rag", "Sparse", 1.0),
    ("exp_04_hybrid_rag", "Hybrid", 1.0),
    ("exp_05_multi_hop_rag", "MultiHop", 3.0),
    ("exp_07_adaptive_variant_a", "Variant A", None),
    ("exp_07_adaptive_variant_b", "Variant B", None),
]:
    sp = REPO_ROOT / f"results/{prefix}__test_1273/summary.json"
    if not sp.exists():
        continue
    s = json.loads(sp.read_text())
    if calls is None:
        v_letter = label.split()[-1]
        bucket_counts_test = pd.Series([Q_TO_BUCKET.get(q) for q in load_medqa_4opt().query("split=='test'").question]).value_counts().to_dict()
        calls = variant_calls_per_q(ROUTING_TABLES[v_letter], bucket_counts_test)
    headline[label] = {
        "Acuuracy": s["Acuuracy"],
        "calls/Q": calls,
    }

table = pd.DataFrame(headline).T.round(4)
print("=== Accuracy vs Groq calls per question (test_1273) ===")
print(table.to_string())

# Pareto frontier — a point is on the frontier if no other point has both
# higher accuracy AND lower (or equal) calls/Q.
def is_dominated(point_label: str, points: dict) -> bool:
    me = points[point_label]
    for other_label, other in points.items():
        if other_label == point_label:
            continue
        # dominated if SOME other point has >= acc AND <= calls AND at least one strict
        if (other["Acuuracy"] >= me["Acuuracy"] and other["calls/Q"] <= me["calls/Q"] and
            (other["Acuuracy"] > me["Acuuracy"] or other["calls/Q"] < me["calls/Q"])):
            return True
    return False

print("\n=== Pareto frontier ===")
for label in headline:
    flag = "DOMINATED" if is_dominated(label, headline) else "frontier"
    print(f"  {label:15s} acc={headline[label]['Acuuracy']:.4f}  calls/Q={headline[label]['calls/Q']:.3f}  -> {flag}")

# Hypothesis verdicts
print("\n=== Hypothesis verdicts ===")
if "Variant A" in headline and "NoRAG" in headline and "MultiHop" in headline:
    A = headline["Variant A"]; N = headline["NoRAG"]; M = headline["MultiHop"]
    h1 = (A["Acuuracy"] > N["Acuuracy"]) and (A["calls/Q"] < M["calls/Q"]) and (not is_dominated("Variant A", headline))
    print(f"  H1 Variant A on Pareto frontier between NoRAG and MultiHop?  {('✓ SUPPORTED' if h1 else '✗ FALSIFIED')}")

if "Variant A" in headline and "Variant B" in headline:
    A = headline["Variant A"]; B = headline["Variant B"]
    # A dominates B iff A.acc >= B.acc AND A.calls <= B.calls AND at least one strict
    h2 = (A["Acuuracy"] >= B["Acuuracy"] and A["calls/Q"] <= B["calls/Q"] and
          (A["Acuuracy"] > B["Acuuracy"] or A["calls/Q"] < B["calls/Q"]))
    print(f"  H2 Variant A dominates Variant B?  {('✓ SUPPORTED' if h2 else '✗ FALSIFIED')}")

if "Variant A" in headline and "NoRAG" in headline and "MultiHop" in headline:
    A = headline["Variant A"]; N = headline["NoRAG"]; M = headline["MultiHop"]
    marginal_a = (A["Acuuracy"] - N["Acuuracy"]) / (A["calls/Q"] - N["calls/Q"]) if A["calls/Q"] > N["calls/Q"] else float("inf")
    marginal_m = (M["Acuuracy"] - A["Acuuracy"]) / (M["calls/Q"] - A["calls/Q"]) if M["calls/Q"] > A["calls/Q"] else float("inf")
    h3 = marginal_a > marginal_m
    print(f"  H3 Variant A's marginal acc/extra-call > MultiHop's marginal acc/extra-call?  {('✓ SUPPORTED' if h3 else '✗ FALSIFIED')}")
    print(f"     Variant A vs NoRAG  : +{(A['Acuuracy']-N['Acuuracy'])*100:.2f} pp  per +{A['calls/Q']-N['calls/Q']:.2f} call/Q  =>  marginal = {marginal_a:.4f} acc/call")
    print(f"     MultiHop vs Variant A: +{(M['Acuuracy']-A['Acuuracy'])*100:.2f} pp  per +{M['calls/Q']-A['calls/Q']:.2f} call/Q  =>  marginal = {marginal_m:.4f} acc/call")
    if h3:
        ratio = marginal_a / marginal_m if marginal_m > 0 else float("inf")
        print(f"     ratio: Variant A is {ratio:.1f}x more marginally efficient than MultiHop on top of A")

=== Accuracy vs Groq calls per question (test_1273) ===
           Acuuracy  calls/Q
NoRAG        0.7738    1.000
Naive        0.7573    1.000
Sparse       0.7581    1.000
Hybrid       0.7659    1.000
MultiHop     0.7958    3.000
Variant A    0.7863    1.806
Variant B    0.7832    2.425

=== Pareto frontier ===
  NoRAG           acc=0.7738  calls/Q=1.000  -> frontier
  Naive           acc=0.7573  calls/Q=1.000  -> DOMINATED
  Sparse          acc=0.7581  calls/Q=1.000  -> DOMINATED
  Hybrid          acc=0.7659  calls/Q=1.000  -> DOMINATED
  MultiHop        acc=0.7958  calls/Q=3.000  -> frontier
  Variant A       acc=0.7863  calls/Q=1.806  -> frontier
  Variant B       acc=0.7832  calls/Q=2.425  -> DOMINATED

=== Hypothesis verdicts ===
  H1 Variant A on Pareto frontier between NoRAG and MultiHop?  ✓ SUPPORTED
  H2 Variant A dominates Variant B?  ✓ SUPPORTED
  H3 Variant A's marginal acc/extra-call > MultiHop's marginal acc/extra-call?  ✓ SUPPORTED
     Variant A vs NoRAG  : +1.26 pp  pe

## 10. Per-bucket accuracy attribution (Table 2 / Table 10 fodder)

In [12]:
# Per-bucket per-variant accuracy = the routed-arch's per-question accuracy, averaged in-bucket.
test_qs = load_medqa_4opt().query("split == 'test'").reset_index(drop=False).rename(columns={"index":"row_idx"})
test_qs["question_id"] = "medqa_" + test_qs.row_idx.astype(str)
test_qs["bucket"] = test_qs.question.map(Q_TO_BUCKET)

def per_bucket_acc(variant_letter: str) -> pd.DataFrame:
    fp = REPO_ROOT / f"results/exp_07_adaptive_variant_{variant_letter.lower()}__test_1273/predictions.jsonl"
    if not fp.exists():
        return pd.DataFrame()
    p = pd.DataFrame([json.loads(line) for line in fp.read_text().splitlines()])
    j = p.merge(test_qs[["question_id","bucket"]], on="question_id")
    return j.groupby("bucket").is_correct.agg(["count","mean"]).rename(columns={"count":"n","mean":"acc"}).round(4)

for v in ("A","B"):
    out = per_bucket_acc(v)
    if not out.empty:
        print(f"\n=== Variant {v} per-bucket accuracy (test_1273) ===")
        print(out.reindex(["Simple","Moderate","Complex"]).to_string())

# Cross-check: simulate using underlying architectures' per-question predictions.
print("\n=== Cross-check: simulator on existing predictions vs actual run ===")
merged = test_qs[["question_id","bucket"]]
for arch_label, prefix in [("NoRAG","exp_01_base_llm"),("Naive","exp_02_naive_rag"),
                            ("Sparse","exp_03_sparse_rag"),("Hybrid","exp_04_hybrid_rag"),
                            ("MultiHop","exp_05_multi_hop_rag")]:
    p = pd.DataFrame([json.loads(l) for l in (REPO_ROOT/f"results/{prefix}__test_1273/predictions.jsonl").read_text().splitlines()])
    merged = merged.merge(p[["question_id","is_correct"]].rename(columns={"is_correct":arch_label}), on="question_id")

for v_label, table_v in ROUTING_TABLES.items():
    sim = merged.apply(lambda r: r[table_v[r.bucket]], axis=1)
    print(f"  Variant {v_label} simulated  acc = {sim.mean():.4f}")


=== Variant A per-bucket accuracy (test_1273) ===
            n     acc
bucket               
Simple    366  0.8197
Moderate  394  0.7589
Complex   513  0.7836

=== Variant B per-bucket accuracy (test_1273) ===
            n     acc
bucket               
Simple    366  0.7951
Moderate  394  0.7716
Complex   513  0.7836

=== Cross-check: simulator on existing predictions vs actual run ===
  Variant A simulated  acc = 0.7895
  Variant B simulated  acc = 0.7840


**The simulated number should equal the actual run within rounding** — if it doesn't, there's either a cache miss bug or a deterministic-retriever drift to investigate.

---

**Next.** EXP_07 done → Phase 5 complete. Tables 2, 3, 10 ready to populate. Move to Phase 6 — passage-level LIME / SHAP (notebooks `06_exp10_*`, `06_exp11_*`, `06_exp12_*`). Multi-Hop is the highest-value architecture to explain (Faithfulness has graded signal there).